Transition-Specific Contextual Analysis: This notebook investigates contextual and spatial factors associated with specific behavioral state transitions

Load contextual transition dataset 

In [ ]:
import pandas as pd

# Load shared project paths and configuration
%run 00_config.ipynb

CONTEXT_FILE = (
    OUTPUT_DIR
    / "contextual_transition_dataset.xlsx"
)

GEO_FILE = (
    SAMSUNG_DIR
    / "Geo_file_Cleaned.csv"
)

context_df = pd.read_excel(CONTEXT_FILE)

Create transition name

In [ ]:
context_df["transition_name"] = (
    context_df["previous_state"]
    + " -> "
    + context_df["new_state"]
)

Keep only real transitions

In [ ]:
real_transitions = context_df[
    context_df["transition"] == 1
].copy()

Count transitions

In [ ]:
transition_counts = (
    real_transitions["transition_name"]
    .value_counts()
    .reset_index()
)

transition_counts.columns = [
    "transition_name",
    "count"
]

transition_counts["percentage"] = (
    transition_counts["count"]
    / transition_counts["count"].sum()
    * 100
).round(2)

transition_counts

Number of users per transition

In [ ]:
transition_users = (
    real_transitions
    .groupby("transition_name")
    .agg(
        n_occurrences=("user_id", "count"),
        n_users=("user_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "n_occurrences",
        ascending=False
    )
)

transition_users

Select the 3 most frequent transitions

In [ ]:
top_3_names = (
    real_transitions["transition_name"]
    .value_counts()
    .nlargest(3)
    .index
    .tolist()
)

micro_df = real_transitions[
    real_transitions["transition_name"].isin(top_3_names)
].copy()

Summary of the selected transitions

In [ ]:
top3_transition_counts = (
    micro_df["transition_name"]
    .value_counts()
    .reset_index()
)

top3_transition_counts.columns = [
    "transition_name",
    "count"
]

top3_transition_counts

User distribution for selected transitions

In [ ]:
micro_users = (
    micro_df
    .groupby("transition_name")
    .agg(
        n_occurrences=("user_id", "count"),
        n_users=("user_id", "nunique")
    )
    .reset_index()
    .sort_values("n_occurrences", ascending=False)
)

micro_users

Create transition event dataset (with day of transition and previous day)

In [ ]:
event_df = micro_df[
    [
        "user_id",
        "date",
        "previous_state",
        "new_state",
        "transition_name"
    ]
].copy()

event_df["transition_day"] = pd.to_datetime(
    event_df["date"]
)

event_df["previous_day"] = (
    event_df["transition_day"]
    - pd.Timedelta(days=1)
)

event_df = event_df.drop(
    columns=["date"]
)

event_df = event_df.sort_values(
    ["transition_name", "transition_day"]
).reset_index(drop=True)

Select contextual variables

In [ ]:
context_variables = [
    "temperature_2m",
    "precipitation",
    "rain_day",
    "social_context",
    "social_presence",
    "n_routes",
    "mean_route_duration",
    "n_space_transitions",
    "is_weekend"
]

context_base = context_df[
    ["user_id", "date"] + context_variables
].copy()

context_base["date"] = pd.to_datetime(
    context_base["date"]
)

Load  geographical dataset

In [ ]:
geo = pd.read_csv(GEO_FILE)

Prepare geographical dataset

In [ ]:
geo["timestamp"] = pd.to_datetime(
    geo["timestamp"]
)

geo["date"] = (
    geo["timestamp"]
    .dt.normalize()
)

Identify space type columns

In [ ]:
space_columns = [
    "forest_green_space",
    "urban_green_space",
    "pa_space",
    "food_drinks_space",
    "nightlife_space",
    "retail_space",
    "cultural_space",
    "university_or_study_space",
    "residential_building",
    "commercial_building",
    "industrial_building",
    "public_building",
    "bus_stop",
    "sidewalks",
    "car_park",
    "street_or_avenue"
]

space_columns = [
    col for col in space_columns
    if col in geo.columns
]

Create daily spatial profile

In [ ]:
# Convert space-type columns from yes/no to 1/0.
# A location may belong to more than one space type.
geo[space_columns] = (
    geo[space_columns]
    .replace({
        "yes": 1,
        "no": 0,
        "Yes": 1,
        "No": 0,
        "YES": 1,
        "NO": 0,
        True: 1,
        False: 0
    })
)

geo[space_columns] = geo[space_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

daily_space_profile = (
    geo
    .groupby(["user_id", "date"])[space_columns]
    .mean()
    .reset_index()
)

Create aggregated spatial indicators

In [ ]:
daily_space_profile["urban_mobility"] = (
    daily_space_profile["street_or_avenue"]
    + daily_space_profile["sidewalks"]
    + daily_space_profile["bus_stop"]
    + daily_space_profile["car_park"]
)

daily_space_profile["residential_space"] = (
    daily_space_profile["residential_building"]
)

daily_space_profile["academic_space"] = (
    daily_space_profile["university_or_study_space"]
)

daily_space_profile["food_space"] = (
    daily_space_profile["food_drinks_space"]
)

daily_space_profile["green_pa_space"] = (
    daily_space_profile["urban_green_space"]
    + daily_space_profile["forest_green_space"]
    + daily_space_profile["pa_space"]
)

daily_space_profile["social_cultural_space"] = (
    daily_space_profile["nightlife_space"]
    + daily_space_profile["cultural_space"]
)

daily_space_profile["public_commercial_space"] = (
    daily_space_profile["public_building"]
    + daily_space_profile["industrial_building"]
    + daily_space_profile["commercial_building"]
    + daily_space_profile["retail_space"]
)

daily_space_profile[
    [
        "urban_mobility",
        "residential_space",
        "academic_space",
        "food_space",
        "green_pa_space",
        "social_cultural_space",
        "public_commercial_space"
    ]
].describe()

Select spatial variables

In [ ]:
spatial_variables = [
    "urban_mobility",
    "residential_space",
    "academic_space",
    "food_space",
    "green_pa_space",
    "social_cultural_space",
    "public_commercial_space"
]

Prepare spatial profile for merge

In [ ]:
daily_space_profile_selected = daily_space_profile[
    ["user_id", "date"] + spatial_variables
].copy()

Get spatial profile for the day before transition

In [ ]:
space_before = daily_space_profile_selected.rename(
    columns={
        "date": "previous_day",
        **{
            var: f"{var}_before"
            for var in spatial_variables
        }
    }
)

event_df_space = event_df.merge(
    space_before,
    on=["user_id", "previous_day"],
    how="left"
)

Get spatial profile of transition day

In [ ]:
space_transition = daily_space_profile_selected.rename(
    columns={
        "date": "transition_day",
        **{
            var: f"{var}_transition"
            for var in spatial_variables
        }
    }
)

event_df_space = event_df_space.merge(
    space_transition,
    on=["user_id", "transition_day"],
    how="left"
)

Compute spatial changes between previous day and transition day

In [ ]:
for variable in spatial_variables:
    event_df_space[f"delta_{variable}"] = (
        event_df_space[f"{variable}_transition"]
        - event_df_space[f"{variable}_before"]
    )

Define delta spatial variables

In [ ]:
delta_spatial_variables = [
    f"delta_{variable}"
    for variable in spatial_variables
]

Check for missing values

In [ ]:
missing_space = (
    event_df_space
    .isna()
    .sum()
    .reset_index()
)

missing_space.columns = [
    "variable",
    "missing_values"
]

missing_space

Check complete events

In [ ]:
complete_events = event_df_space.dropna(
    subset=delta_spatial_variables
)

print("Total events:", len(event_df_space))
print("Complete events:", len(complete_events))
print("Missing events:", len(event_df_space) - len(complete_events))

Available events per transition

In [ ]:
complete_events["transition_name"].value_counts()

Mean and Median of spatial change 

In [ ]:
mean_spatial_change = (
    complete_events
    .groupby("transition_name")[delta_spatial_variables]
    .mean()
)

median_spatial_change = (
    complete_events
    .groupby("transition_name")[delta_spatial_variables]
    .median()
)

comparison = pd.concat(
    [
        mean_spatial_change.add_suffix("_mean"),
        median_spatial_change.add_suffix("_median")
    ],
    axis=1
)

comparison

Add contextual values to transition events

In [ ]:
event_context = event_df.merge(
    context_base,
    left_on=["user_id", "transition_day"],
    right_on=["user_id", "date"],
    how="left"
)

event_context.head()


Contextual profile for each transition

In [ ]:
context_summary_variables = [
    "temperature_2m",
    "precipitation",
    "social_presence",
    "n_routes",
    "mean_route_duration",
    "n_space_transitions"
]

for transition in top_3_names:
    print("=" * 80)
    print(transition)
    print("=" * 80)

    temp = event_context[
        event_context["transition_name"] == transition
    ]

    print(
        temp[
            context_summary_variables
        ].describe()
    )

IA -> RA vs IA -> IA

In [ ]:
# Transition group
ia_ra = context_df[
    context_df["transition_name"]
    == "Irregularly Active -> Regularly Active"
].copy()

print("IA -> RA:", len(ia_ra))

# Control group
ia_ia = context_df[
    (
        context_df["previous_state"]
        == "Irregularly Active"
    )
    &
    (
        context_df["new_state"]
        == "Irregularly Active"
    )
].copy()

print("IA -> IA:", len(ia_ia))

Descriptive statistics

In [ ]:
ia_ra["group"] = "transition"
ia_ia["group"] = "control"

ia_analysis = pd.concat(
    [ia_ra, ia_ia],
    ignore_index=True
)

continuous_variables = [
    "temperature_2m",
    "precipitation",
    "n_routes",
    "mean_route_duration",
    "n_space_transitions"
]

ia_descriptive = (
    ia_analysis
    .groupby("group")[continuous_variables]
    .agg(["count", "mean", "median", "std"])
)

ia_descriptive

Mann-Whitney

In [ ]:
from scipy.stats import mannwhitneyu

ia_mann_whitney_results = []

for variable in continuous_variables:

    transition = ia_analysis[
        ia_analysis["group"] == "transition"
    ][variable].dropna()

    control = ia_analysis[
        ia_analysis["group"] == "control"
    ][variable].dropna()

    statistic, p_value = mannwhitneyu(
        transition,
        control,
        alternative="two-sided"
    )

    ia_mann_whitney_results.append({
        "variable": variable,
        "median_transition": transition.median(),
        "median_control": control.median(),
        "p_value": p_value
    })

ia_mann_whitney = pd.DataFrame(
    ia_mann_whitney_results
)

ia_mann_whitney

RA -> IA vs RA -> RA

In [ ]:
# Transition group
ra_ia = context_df[
    context_df["transition_name"]
    == "Regularly Active -> Irregularly Active"
].copy()

# Control group
ra_ra = context_df[
    (
        context_df["previous_state"] == "Regularly Active"
    )
    &
    (
        context_df["new_state"] == "Regularly Active"
    )
].copy()

print("RA -> IA:", len(ra_ia))
print("RA -> RA:", len(ra_ra))

Prepare Comparinson Groups

In [ ]:
ra_ia["group"] = "transition"
ra_ra["group"] = "control"

ra_analysis = pd.concat(
    [ra_ia, ra_ra],
    ignore_index=True
)

Descriptive statistics for RA -> IA versus RA -> RA

In [ ]:
ra_descriptive = (
    ra_analysis
    .groupby("group")[continuous_variables]
    .agg(["count", "mean", "median", "std"])
)

ra_descriptive

Mann-Whitney tests for RA -> IA versus RA -> RA

In [ ]:
ra_mann_whitney_results = []

for variable in continuous_variables:

    transition = ra_analysis[
        ra_analysis["group"] == "transition"
    ][variable].dropna()

    control = ra_analysis[
        ra_analysis["group"] == "control"
    ][variable].dropna()

    statistic, p_value = mannwhitneyu(
        transition,
        control,
        alternative="two-sided"
    )

    ra_mann_whitney_results.append({
        "variable": variable,
        "median_transition": transition.median(),
        "median_control": control.median(),
        "p_value": p_value
    })

ra_mann_whitney = pd.DataFrame(
    ra_mann_whitney_results
)

ra_mann_whitney